## Tools

## Runtime

Runtime is used to access the state. Via runtime we can access the messages, 

In [9]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import HumanMessage

load_dotenv()

class Answer(BaseModel):
    answer: str
    confidence: float

@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent user message"""
    messages = runtime.state["messages"]
    
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content
    
    return "No user message found."

model = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)

llm = create_agent(
    model = model,
    tools = [get_last_user_message],
    system_prompt = "You are a helpful assistant that can access the last user message using the get_last_user_message tool.",
    # response_format = Answer
)

result = llm.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What was the last message I sent?"
        }
    ]
})

print(result["messages"][-1].content_blocks)

    

[{'type': 'reasoning', 'reasoning': 'The user asks: "What was the last message I sent?" We need to retrieve the last user message. The tool get_last_user_message returns the most recent user message. The last user message is the current one? Actually the conversation: system, developer, user, assistant (we just called tool), tool returned the same message? The tool returned the content: "What was the last message I sent?" That\'s the last user message. So answer: The last message you sent was "What was the last message I sent?"'}, {'type': 'text', 'text': 'Your most recent message was:\n\n**“What was the last message I sent?”**'}]


In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

model = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"


llm = create_agent(
    model = model,
)

llm.bind_tools(search_database)


